# ДЗ 4. Olist: typed-схема, DML з RETURNING / ON CONFLICT, JOIN, агрегати, set operations

**Датасет:** Brazilian E-Commerce Public Dataset by Olist ([Kaggle: olistbr/brazilian-ecommerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)).
**Потік:** CSV → `*_raw` (проміжний шар) → typed-таблиці з constraints → DML → JOIN/агрегати → set operations/window → Reflection.
Усі висновки під запитами формуються з фактичного результату, тому лишаються правильними після *Restart & Run All*.

## 1. Setup і typed-схема Olist

In [ ]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
from pathlib import Path

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import Markdown, display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

pg = pgserver.get_server("/tmp/hw4_pg", cleanup_mode="stop")
# явно psycopg2, щоб поведінка не залежала від версії SQLAlchemy
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def run(sql: str) -> pd.DataFrame:
    """SELECT -> DataFrame."""
    return pd.read_sql(text(sql), engine)


def run_dml(sql: str) -> pd.DataFrame:
    """INSERT/UPDATE/DELETE ... RETURNING у транзакції з commit; повертає RETURNING як DataFrame."""
    with engine.begin() as conn:
        res = conn.execute(text(sql))
        return pd.DataFrame(res.fetchall(), columns=list(res.keys())) if res.returns_rows else pd.DataFrame()


def execute(sql: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(sql))



def show(t: str) -> None:
    display(Markdown(t))


def n(x) -> str:
    return f"{int(x):,}".replace(",", " ")

### 1.1. Отримання файлів

Пріоритет: CSV у `data/olist/` (курсовий архів, див. README); якщо їх немає — завантаження з Kaggle через `kagglehub` (потрібні налаштовані Kaggle credentials).

In [ ]:
LOCAL_DIR = Path("data/olist")
if (LOCAL_DIR / "olist_orders_dataset.csv").exists():
    dataset_dir = str(LOCAL_DIR)                      # Варіант B: локальні CSV
else:
    import kagglehub                                   # Варіант A: Kaggle
    dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("dataset_dir =", dataset_dir)

files = {
    "olist_customers_raw":     "olist_customers_dataset.csv",
    "olist_orders_raw":        "olist_orders_dataset.csv",
    "olist_order_items_raw":   "olist_order_items_dataset.csv",
    "olist_products_raw":      "olist_products_dataset.csv",
    "olist_sellers_raw":       "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

raw_counts = {}
for table_name, file_name in files.items():
    df = pd.read_csv(os.path.join(dataset_dir, file_name))
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=5_000, method="multi")
    raw_counts[table_name] = len(df)
    print(table_name, df.shape)

### 1.2. Typed-таблиці

Raw-таблиці — лише проміжний шар (усе, що вивів `pandas`, без constraints). Усі подальші завдання працюють з typed-таблицями: `PRIMARY KEY`/`FOREIGN KEY` гарантують цілісність, `CHECK` — допустимі статуси й діапазони, `NUMERIC(12,2)` — точні грошові значення, `TIMESTAMP` — дати замість рядків.

In [ ]:
execute("""
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);
CREATE INDEX idx_olist_customers_unique_id ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN ('created','approved','invoiced','processing',
                            'shipped','delivered','unavailable','canceled'))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
CREATE INDEX idx_olist_order_items_seller_order ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);
CREATE INDEX idx_olist_order_reviews_order_id ON olist_order_reviews (order_id);
""")
print("typed-схему створено")

### 1.3. `INSERT ... SELECT` raw → typed

У вихідних CSV колонки названо з помилкою `product_name_lenght` / `product_description_lenght`: у `SELECT` звертаємось до **фактичних** назв, а в typed-схемі вони нормалізовані (`..._length`). Порядок вставки відповідає залежностям FK: customers → orders, products/sellers → order_items, orders → reviews. Явні списки колонок захищають від тихого зсуву, якщо порядок колонок у CSV зміниться.

In [ ]:
execute("""
INSERT INTO olist_customers (customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state)
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders (order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at,
                          order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date)
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP, order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP, order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products (product_id, product_category_name, product_name_length, product_description_length,
                            product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm)
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER, product_description_lenght::INTEGER,
       product_photos_qty::INTEGER, product_weight_g::INTEGER,
       product_length_cm::INTEGER, product_height_cm::INTEGER, product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers (seller_id, seller_zip_code_prefix, seller_city, seller_state)
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items (order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value)
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP, price::NUMERIC(12, 2), freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (review_id, order_id, review_score, review_comment_title, review_comment_message,
                                 review_creation_date, review_answer_timestamp)
SELECT review_id, order_id, review_score::SMALLINT, review_comment_title, review_comment_message,
       review_creation_date::TIMESTAMP, review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;
""")
print("raw -> typed виконано")

### 1.4. Smoke-test кількості рядків

In [ ]:
smoke = run("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders',      COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products',    COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers',     COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews',     COUNT(*) FROM olist_order_reviews
ORDER BY table_name;
""")
raw_map = {"customers": "olist_customers_raw", "orders": "olist_orders_raw", "order_items": "olist_order_items_raw",
           "products": "olist_products_raw", "sellers": "olist_sellers_raw", "reviews": "olist_order_reviews_raw"}
smoke["raw_rows"] = smoke.table_name.map(lambda t: raw_counts[raw_map[t]])
smoke["diff"] = smoke.raw_rows - smoke.n_rows
display(smoke)
bad = smoke[smoke["diff"] != 0]
if bad.empty:
    show("**Висновок.** У всіх шести таблицях кількість рядків typed = raw: жоден рядок не втрачено, а всі `FOREIGN KEY` і `CHECK` пройшли без відхилень (інакше `INSERT` завершився б помилкою, бо тут немає фільтрації).")
else:
    show("**Висновок.** Розбіжності: " + "; ".join(f"`{r.table_name}` (raw {n(r.raw_rows)}, typed {n(r.n_rows)})" for r in bad.itertuples()))

## 2. DML-запити з PostgreSQL-фічами

### 2.1. `INSERT ... RETURNING`: alert для продавця з найбільшою кількістю запізнень

**Що змінює:** додає один рядок у `seller_alerts`. **`RETURNING`** повертає згенеровані сервером значення (`alert_id` з identity, `created_at` з `DEFAULT NOW()`) без другого `SELECT` — отриманий рядок одразу є audit-записом того, що саме було створено. Клітинка починається з `DROP TABLE`/`CREATE TABLE`, тому повторний запуск не накопичує дублікатів.

In [ ]:
execute("""
DROP TABLE IF EXISTS seller_alerts CASCADE;
CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")
alert = run_dml("""
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT so.seller_id, COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT seller_id, 'late_delivery', 4,
       'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;
""")
display(alert)
a = alert.iloc[0]
show(f"**Interpretation:** Створено alert №{a.alert_id} для продавця `{a.seller_id}` — {a.details}. "
     "`DISTINCT` у `seller_orders` гарантує, що замовлення з кількома позиціями одного продавця рахується один раз. "
     "Результат `RETURNING` — готовий запис для audit-логу в notebook: видно і кого, і з яким `alert_id`/часом було створено.")

### 2.2. `INSERT ... ON CONFLICT DO UPDATE`: idempotent upsert seller-score

**Чому `seller_orders` з `SELECT DISTINCT`.** `olist_order_items` має грануляцію *замовлення × позиція*; якщо продавець має кілька позицій в одному замовленні, без `DISTINCT` те саме замовлення (а разом із ним і review) потрапило б у розрахунок кілька разів. `DISTINCT (seller_id, order_id)` зводить грануляцію до *seller × order*.

**Чому review спочатку агрегується до `order_id`.** Замовлення може мати більше одного review-запису, а join `order_items → reviews` множить кожен відгук на кількість позицій. Тому спершу `review_per_order` (один рядок на замовлення), потім seller-метрики.

**Чому `ON CONFLICT (seller_id) DO UPDATE` робить запуск повторюваним.** Повторний запуск не додає рядків, а перезаписує існуючі значення для того самого `seller_id` (і оновлює `updated_at`) — стан після n запусків той самий, що й після одного.

**Чому sellers без review мають `avg_rating = NULL`, `n_reviews = 0`, `tier = NULL`.** `LEFT JOIN` залишає продавця без відгуків; `AVG` по порожній множині дає `NULL`, `COUNT(rpo.order_id)` рахує лише не-NULL (отже 0), а `CASE` на `NULL` не присвоює жодного tier — «немає даних» не маскується під «поганий рейтинг».

**Обмеження датасету.** Review в Olist прив’язаний до *замовлення*, а не до продавця: якщо замовлення містить товари кількох продавців, один order-level відгук зараховується кожному з них. Це прийнятне навчальне спрощення.

In [ ]:
# Демонстрація помилки: наївний join множить відгуки через позиції замовлення
cmp = run("""
WITH naive AS (
    SELECT COUNT(rv.review_row_id) AS n_reviews_naive
    FROM olist_order_items AS oi
    LEFT JOIN olist_order_reviews AS rv ON rv.order_id = oi.order_id
),
correct AS (
    SELECT COUNT(rpo.order_id) AS n_reviews_correct
    FROM (SELECT DISTINCT seller_id, order_id FROM olist_order_items) AS so
    LEFT JOIN (SELECT order_id FROM olist_order_reviews GROUP BY order_id) AS rpo ON rpo.order_id = so.order_id
)
SELECT n_reviews_naive, n_reviews_correct, n_reviews_naive - n_reviews_correct AS inflation FROM naive, correct;
""")
display(cmp)
c = cmp.iloc[0]
show(f"**Перевірка грануляції.** Наївний join порахував {n(c.n_reviews_naive)} відгуків проти коректних {n(c.n_reviews_correct)}: завищення на {n(c.inflation)} ({c.inflation*100/c.n_reviews_correct:.1f}%) — саме через кілька позицій в одному замовленні.")

In [ ]:
UPSERT_SCORE = """
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT seller_id, avg_rating, n_reviews,
       CASE WHEN avg_rating IS NULL THEN NULL
            WHEN avg_rating >= 4.5 THEN 'gold'
            WHEN avg_rating >= 3.5 THEN 'silver'
            ELSE 'bronze' END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
"""
CTE_SCORE = """
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT so.seller_id,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
           COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
"""
execute("""
DROP TABLE IF EXISTS seller_score CASCADE;
CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
""")

def upsert_score():
    # RETURNING використано в агрегованому вигляді: повний вивід (тисячі продавців) не потрібен
    return run_dml(CTE_SCORE.rstrip() + ", ups AS (" + UPSERT_SCORE.strip() +
                   " RETURNING seller_id, avg_rating, n_reviews, tier, updated_at) "
                   "SELECT COALESCE(tier, '(no reviews)') AS tier, COUNT(*) AS n_sellers, "
                   "ROUND(AVG(avg_rating), 2) AS mean_rating, SUM(n_reviews) AS total_reviews "
                   "FROM ups GROUP BY tier ORDER BY tier;")

first = upsert_score()
cnt1 = run("SELECT COUNT(*) AS c FROM seller_score").c[0]
second = upsert_score()                       # повторний запуск
cnt2 = run("SELECT COUNT(*) AS c FROM seller_score").c[0]
display(first)
assert cnt1 == cnt2, "upsert має бути ідемпотентним"
no_rev = run("SELECT COUNT(*) AS c FROM seller_score WHERE avg_rating IS NULL AND n_reviews = 0 AND tier IS NULL").c[0]
show(f"**Interpretation:** Після першого запуску в `seller_score` {n(cnt1)} рядків, після повторного — {n(cnt2)}: **дублікатів немає**, `ON CONFLICT` просто оновив рядки. "
     f"Розподіл tier: " + "; ".join(f"{r.tier} — {n(r.n_sellers)}" for r in first.itertuples()) + f". Продавців без жодного review (NULL / 0 / NULL): {n(no_rev)}. "
     "Пам’ятаємо про спрощення: order-level відгук приписується кожному продавцю замовлення.")

### 2.3. `UPDATE ... FROM`: прапорець запізнення `is_late`

**Що змінює:** додає в `olist_orders` колонку `is_late` і перераховує її з дат. Запит **ідемпотентний**: значення завжди виводиться з поточних дат (а не «дописується TRUE»), тому повторний запуск дає той самий результат. `RETURNING` тут згорнуто в агрегат — для 100 тис. рядків друкувати кожен не потрібно, але саме він дає audit: скільки рядків оновлено й з яким значенням.

In [ ]:
execute("ALTER TABLE olist_orders ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;")
UPD = """
WITH delivery_flags AS (
    SELECT order_id,
           (order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE) AS is_late_calc
    FROM olist_orders
),
upd AS (
    UPDATE olist_orders AS o
    SET is_late = f.is_late_calc
    FROM delivery_flags AS f
    WHERE f.order_id = o.order_id
    RETURNING o.order_id, o.customer_id, o.is_late
)
SELECT is_late, COUNT(*) AS n_updated FROM upd GROUP BY is_late ORDER BY is_late;
"""
u1 = run_dml(UPD)
state1 = run("SELECT COUNT(*) FILTER (WHERE is_late) AS late FROM olist_orders").late[0]
u2 = run_dml(UPD)
state2 = run("SELECT COUNT(*) FILTER (WHERE is_late) AS late FROM olist_orders").late[0]
display(u1)
assert state1 == state2
tot = u1.n_updated.sum()
show(f"**Interpretation:** Оновлено {n(tot)} замовлень; запізнилось (`delivered > estimated` за датою) {n(state1)} ({state1*100/tot:.2f}%). "
     f"Повторний запуск дав ту саму кількість ({n(state2)}) — запит ідемпотентний. Замовлення без фактичної дати доставки (скасовані, у дорозі) отримують `FALSE`, "
     "тобто `is_late = FALSE` означає «не доведено запізнення», а не «доставлено вчасно».")

### 2.4. `DELETE ... RETURNING`: контрольоване видалення застарілих alert-ів

Alert із п.2.1 свіжий, тому запит `DELETE ... WHERE created_at < NOW() - INTERVAL '30 days'` на «чистій» таблиці **повернув би порожній результат** — це нормально: застарілих записів ще немає. Щоб показати механізм, спершу вставляємо один явно позначений демонстраційний alert із датою 45 днів тому, а потім виконуємо `DELETE`; `RETURNING` залишає в notebook слід того, що саме видалено. Свіжий alert із 2.1 не чіпається.

In [ ]:
run_dml("""
INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
SELECT seller_id, 'data_quality', 1, 'DEMO stale alert (to be deleted)', NOW() - INTERVAL '45 days'
FROM olist_sellers ORDER BY seller_id LIMIT 1
RETURNING alert_id;
""")
deleted = run_dml("""
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, details, created_at;
""")
display(deleted)
left = run("SELECT alert_id, alert_type, severity FROM seller_alerts ORDER BY alert_id")
display(left)
again = run_dml("DELETE FROM seller_alerts WHERE created_at < NOW() - INTERVAL '30 days' RETURNING alert_id")
show(f"**Interpretation:** Видалено застарілих alert-ів: {len(deleted)} (`RETURNING` показав їхні `alert_id`, продавця й час створення); у таблиці лишилось {len(left)} актуальних. "
     f"Повторний запуск повернув {len(again)} рядків — операція безпечна для повторення: другий раз просто нічого не знаходить.")

## 3. `customer_segments`: сегментація на рівні `customer_unique_id`

**Чому `customer_unique_id`.** У Olist `customer_id` генерується на *кожне замовлення*, тому за ним frequency майже завжди дорівнює 1, а повторні покупки одного покупця розпадаються на різні «клієнти». `customer_unique_id` ідентифікує реальну людину.

**Чому потрібна `olist_customer_dim`.** `customer_unique_id` не унікальний в `olist_customers` (там багато рядків на одну людину), тому на нього не можна поставити `FOREIGN KEY`. Дедуплікований dimension (`DISTINCT ON` — по одному рядку на покупця, найновіший `customer_id`) має `PRIMARY KEY` на `customer_unique_id` і дає коректну ціль для FK.

In [ ]:
execute("""
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (customer_unique_id, latest_customer_id, customer_state, customer_city)
SELECT DISTINCT ON (c.customer_unique_id)
       c.customer_unique_id, c.customer_id AS latest_customer_id, c.customer_state, c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o ON o.customer_id = c.customer_id
ORDER BY c.customer_unique_id, o.order_purchase_timestamp DESC NULLS LAST, c.customer_id;

CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),
    rfm_score           SMALLINT NOT NULL CHECK (rfm_score BETWEEN 1 AND 5),
    recency_days        INTEGER CHECK (recency_days IS NULL OR recency_days >= 0),
    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0 CHECK (monetary_value >= 0),
    n_orders            INTEGER NOT NULL DEFAULT 0 CHECK (n_orders >= 0),
    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;
""")
dim = run("""SELECT (SELECT COUNT(*) FROM olist_customers) AS customer_id_rows,
                    (SELECT COUNT(*) FROM olist_customer_dim) AS unique_customers""")
display(dim)
d = dim.iloc[0]
show(f"**Інтерпретація.** У `olist_customers` {n(d.customer_id_rows)} рядків (по одному на замовлення), а реальних покупців {n(d.unique_customers)}: "
     f"{n(d.customer_id_rows - d.unique_customers)} рядків — це повторні покупки тих самих людей. Саме їх втратив би RFM за `customer_id`.")

### 3.1. Наповнення через `INSERT ... SELECT ... ON CONFLICT`

**Чому `reference_date` береться з датасету.** Olist — *історичний* знімок; `CURRENT_DATE` зробив би `recency_days` величезним і щодня різним, а результат — нестабільним. `MAX(order_purchase_timestamp) + 1 день` — детермінована «точка відліку», тож сегментація відтворювана.

**Як `ON CONFLICT` робить сегментацію повторюваною.** `UNIQUE (customer_unique_id)` є ціллю конфлікту: другий запуск оновлює наявні рядки (сегмент, score, `assigned_at`), а не додає нові. Сегментація — спрощена: `rfm_score` комбінує monetary і frequency (recency використовується в сегменті `new`).

In [ ]:
SEG_CTE = """
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date FROM olist_orders
),
order_totals AS (
    SELECT o.order_id, o.customer_id, o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, o.customer_id, o.order_purchase_timestamp
),
customer_stats AS (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
           MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT cs.customer_unique_id, cs.n_orders, cs.monetary_value,
           CASE WHEN cs.last_order_date IS NULL THEN NULL
                ELSE (rd.as_of_date - cs.last_order_date)::INTEGER END AS recency_days,
           CASE WHEN cs.monetary_value >= 1000 THEN 'VIP'
                WHEN cs.last_order_date IS NOT NULL
                     AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                     AND cs.n_orders = 1 THEN 'new'
                WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
                ELSE 'inactive' END AS segment_name,
           CASE WHEN cs.monetary_value >= 1000 THEN 5
                WHEN cs.monetary_value >= 500  THEN 4
                WHEN cs.n_orders >= 2          THEN 3
                WHEN cs.n_orders = 1           THEN 2
                ELSE 1 END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
),
ins AS (
    INSERT INTO customer_segments (customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders)
    SELECT customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders FROM segmented
    ON CONFLICT (customer_unique_id) DO UPDATE
    SET segment_name = EXCLUDED.segment_name, rfm_score = EXCLUDED.rfm_score,
        recency_days = EXCLUDED.recency_days, monetary_value = EXCLUDED.monetary_value,
        n_orders = EXCLUDED.n_orders, assigned_at = NOW()
    RETURNING customer_unique_id, segment_name, rfm_score, recency_days, monetary_value, n_orders
)
SELECT segment_name, COUNT(*) AS n_customers,
       ROUND(AVG(monetary_value), 2) AS avg_monetary, ROUND(AVG(n_orders), 2) AS avg_orders,
       ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM ins GROUP BY segment_name ORDER BY n_customers DESC;
"""
seg1 = run_dml(SEG_CTE)
c1 = run("SELECT COUNT(*) AS c FROM customer_segments").c[0]
seg2 = run_dml(SEG_CTE)                        # повторний запуск
c2 = run("SELECT COUNT(*) AS c FROM customer_segments").c[0]
display(seg1)
assert c1 == c2 == run("SELECT COUNT(*) AS c FROM olist_customer_dim").c[0]
top = seg1.iloc[0]
show(f"**Interpretation:** Сегментовано {n(c1)} покупців; після повторного запуску рядків {n(c2)} — дублікатів немає. "
     f"Найбільший сегмент — `{top.segment_name}` ({n(top.n_customers)}, {top.n_customers*100/c1:.1f}%). "
     + "; ".join(f"`{r.segment_name}`: {n(r.n_customers)} покупців, середня сума витрат {r.avg_monetary}, у середньому {r.avg_orders} замовлень" for r in seg1.itertuples())
     + (". Середня кількість замовлень на покупця " + f"{(seg1.avg_orders.astype(float) * seg1.n_customers).sum() / seg1.n_customers.sum():.2f}" +
       (" — повторні покупки рідкісні, тому frequency слабко розрізняє покупців (важливо для RFM/CLV)." if (seg1.avg_orders.astype(float) * seg1.n_customers).sum() / seg1.n_customers.sum() < 1.5 else " — повторні покупки помітні, frequency інформативна.")))

In [ ]:
cons = run("""
SELECT conname, contype, pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;
""")
display(cons)
kinds = {"p": "PRIMARY KEY", "u": "UNIQUE", "f": "FOREIGN KEY", "c": "CHECK"}
cnt = cons.contype.map(kinds).value_counts()
show("**Висновок.** Знайдено обмеження: " + ", ".join(f"{k} × {v}" for k, v in cnt.items()) +
     ". `NOT NULL`/`DEFAULT` у `pg_constraint` для звичайних колонок не показуються — вони видні в `information_schema.columns`.")

## 4. JOIN-запити з business-interpretation

Покриття: INNER JOIN на 4 таблицях (4.1), LEFT JOIN + COALESCE (4.2), FULL OUTER JOIN (4.3), SELF JOIN (4.4), anti-join через `LEFT JOIN ... IS NULL` (4.5) і `NOT EXISTS` (4.6).

**4.1. Business-question:** Які категорії товарів і штати продавців дають найбільший revenue у delivered-замовленнях?

In [ ]:
q41 = run("""
SELECT p.product_category_name, s.seller_state,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders   AS o ON o.order_id   = oi.order_id
JOIN olist_products AS p ON p.product_id = oi.product_id
JOIN olist_sellers  AS s ON s.seller_id  = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST
LIMIT 10;
""")
display(q41)
t = q41.iloc[0]
states = q41.seller_state.value_counts()
show(f"**Interpretation:** Найбільший revenue — категорія `{t.product_category_name}` у штаті продавця {t.seller_state}: {t.revenue} за {n(t.n_orders)} замовлень. "
     f"У топ-10 штат {states.index[0]} трапляється {states.iloc[0]} раз(и) — {'доходи сконцентровані у кількох регіонах продавців' if states.iloc[0] >= 5 else 'концентрація помірна'}. "
     "`INNER JOIN` відкидає позиції без товару чи продавця, а `product_category_name` може бути NULL — такі позиції згруповані окремо.")

**4.2. Business-question:** Які customer-level features можна побудувати для реального покупця на рівні `customer_unique_id`?

In [ ]:
q42 = run("""
WITH order_totals AS (
    SELECT o.order_id, c.customer_unique_id,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.order_id, c.customer_unique_id
),
review_per_order AS (
    SELECT rv.order_id, AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT ot.customer_unique_id,
           COUNT(DISTINCT ot.order_id) AS n_orders,
           COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
           AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT cd.customer_unique_id, cd.customer_state,
       COALESCE(cs.n_orders, 0) AS n_orders,
       COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
       COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score,
       (cs.avg_review_score IS NOT NULL) AS has_review
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST, cd.customer_unique_id
LIMIT 20;
""")
display(q42)
t = q42.iloc[0]
show(f"**Interpretation:** Найбільший spend у вибірці — {t.total_spend} ({n(t.n_orders)} зам.), штат {t.customer_state}; у топ-20 покупців з кількома замовленнями: {n((q42.n_orders > 1).sum())}. "
     "`review_per_order` спершу зводить відгуки до одного рядка на замовлення, тож score не множиться через позиції замовлення. "
     "`COALESCE(..., 0)` для `avg_review_score` — **sentinel**: 0 не є валідною оцінкою (1–5), тому додано `has_review`, щоб модель відрізняла «немає відгуку» від «низької оцінки».")

**4.3. Business-question:** Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [ ]:
q43 = run("""
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers FROM olist_customers GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers FROM olist_sellers GROUP BY seller_state
)
SELECT COALESCE(c.state, s.state) AS state, c.n_customers, s.n_sellers,
       CASE WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
            WHEN c.state IS NOT NULL THEN 'customers_only'
            ELSE 'sellers_only' END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s ON s.state = c.state
ORDER BY state;
""")
display(q43)
pres = q43.state_presence.value_counts()
co = q43[q43.state_presence == "customers_only"].state.tolist()
so = q43[q43.state_presence == "sellers_only"].state.tolist()
show(f"**Interpretation:** Усього {len(q43)} штатів: у обох групах — {pres.get('both', 0)}, лише серед клієнтів — {pres.get('customers_only', 0)}" + (f" ({', '.join(co)})" if co else "") +
     f", лише серед продавців — {pres.get('sellers_only', 0)}" + (f" ({', '.join(so)})" if so else "") + ". " +
     (f"Штати «customers_only» — попит без локальної пропозиції: замовлення звідти доставляються з інших штатів, що може впливати на `freight_value` і строки. " if co else "Усі штати клієнтів мають і локальних продавців. ") +
     "Саме `FULL OUTER JOIN` показує обидва типи розбіжностей одним запитом.")

**4.4. Business-question:** Які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [ ]:
q44 = run("""
SELECT s1.seller_id AS seller_a, s2.seller_id AS seller_b, s1.seller_state,
       s1.seller_city AS city_a, s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
  ON s1.seller_state = s2.seller_state
 AND s1.seller_id < s2.seller_id
 AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;
""")
display(q44)
npairs = run("""
SELECT COUNT(*) AS c FROM olist_sellers s1 JOIN olist_sellers s2
  ON s1.seller_state = s2.seller_state AND s1.seller_id < s2.seller_id
 AND s1.seller_city IS DISTINCT FROM s2.seller_city""").c[0]
show(f"**Interpretation:** Усього {n(npairs)} унікальних пар продавців з одного штату, але з різних міст (показано 20). "
     "Умова `s1.seller_id < s2.seller_id` прибирає дзеркальні пари (A,B)/(B,A) і пари продавця із самим собою; `IS DISTINCT FROM` коректно обробляє NULL у місті. "
     "Кількість пар росте квадратично, тому для реального порівняння їх варто звужувати (наприклад, лише продавці з достатнім обсягом замовлень).")

**4.5. Business-question:** Які замовлення не мають жодного review-запису? (anti-join через `LEFT JOIN ... IS NULL`)

In [ ]:
q45 = run("""
SELECT o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp, o.order_id
LIMIT 20;
""")
display(q45)
nr = run("""
SELECT COUNT(*) AS total, COUNT(*) FILTER (WHERE rv.review_row_id IS NULL) AS without_review
FROM (SELECT DISTINCT o.order_id, rv.review_row_id
      FROM olist_orders o LEFT JOIN olist_order_reviews rv ON rv.order_id = o.order_id) rv""")
by_status = run("""
SELECT o.order_status, COUNT(*) AS n_orders
FROM olist_orders AS o LEFT JOIN olist_order_reviews AS rv ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL GROUP BY o.order_status ORDER BY n_orders DESC""")
display(by_status)
tot_no = by_status.n_orders.sum() if len(by_status) else 0
show(f"**Interpretation:** Замовлень без жодного review: {n(tot_no)}" + (f"; найчастіший статус — `{by_status.order_status[0]}` ({n(by_status.n_orders[0])})." if len(by_status) else ".") +
     " Для ML це означає, що `review_score` не можна вважати відомою ознакою для кожного замовлення: пропуск потрібно кодувати окремо, а не заповнювати нулем.")

**4.6. Business-question:** Які продавці не мають жодної позиції в замовленнях? (anti-join через `NOT EXISTS`)

In [ ]:
q46 = run("""
SELECT s.seller_id, s.seller_state, s.seller_city
FROM olist_sellers AS s
WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS oi WHERE oi.seller_id = s.seller_id)
ORDER BY s.seller_id
LIMIT 20;
""")
display(q46)
tot = run("""SELECT COUNT(*) AS c FROM olist_sellers s
WHERE NOT EXISTS (SELECT 1 FROM olist_order_items oi WHERE oi.seller_id = s.seller_id)""").c[0]
show(f"**Interpretation:** Продавців без жодної проданої позиції: {n(tot)}. " +
     ("Усі продавці з довідника мають продажі, тож таблиця `olist_sellers` не містить «мертвих» записів. " if tot == 0 else "Такі продавці не потраплять у `seller_score` через `INNER`-логіку, але мають лишатися в dimension. ") +
     "`NOT EXISTS` безпечніший за `NOT IN`: він не ламається, якщо підзапит поверне NULL.")

## 5. GROUP BY, HAVING і агрегати

**WHERE vs HAVING.** `WHERE` фільтрує *рядки до агрегації* (у 5.2 — виключає скасовані замовлення ще до підрахунку), `HAVING` фільтрує *групи після агрегації* (у 5.1 — категорії із сумарним revenue > 10 000, у 5.2 — штати з ≥ 100 замовлень). Умову на агрегат (`SUM(...)`, `COUNT(*)`) у `WHERE` написати не можна.

**5.1. Business-question:** Які категорії мають gross revenue понад 10 000?

In [ ]:
q51 = run("""
SELECT p.product_category_name,
       COUNT(DISTINCT oi.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
       SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
       AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;
""")
display(q51.head(15))
tot_rev = q51.gross_revenue.sum()
top3 = q51.head(3)
show(f"**Interpretation:** Поріг 10 000 пройшли {len(q51)} категорій. Лідер — `{q51.product_category_name[0]}` ({q51.gross_revenue[0]}); "
     f"топ-3 дають {top3.gross_revenue.sum()*100/tot_rev:.1f}% revenue серед відфільтрованих категорій. "
     f"Середня ціна позиції варіює від {q51.avg_item_price.min()} до {q51.avg_item_price.max()}: високий revenue може бути наслідком обсягу, а не дорогих товарів.")

**5.2. Business-question:** Яка середня вартість замовлення (AOV) по штатах клієнтів, де є щонайменше 100 нескасованих замовлень?

In [ ]:
q52 = run("""
WITH order_totals AS (
    SELECT o.order_id, c.customer_state,
           SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    WHERE o.order_status <> 'canceled'
    GROUP BY o.order_id, c.customer_state
)
SELECT customer_state, COUNT(*) AS n_orders,
       SUM(order_total)::NUMERIC(12, 2) AS gmv,
       (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;
""")
display(q52)
show(f"**Interpretation:** У вибірці {len(q52)} штатів із ≥ 100 замовлень. Найвищий AOV — {q52.customer_state[0]} ({q52.aov[0]}), найнижчий — {q52.customer_state.iloc[-1]} ({q52.aov.iloc[-1]}). "
     f"Найбільший GMV має {q52.sort_values('gmv', ascending=False).customer_state.iloc[0]}. Високий AOV при малому числі замовлень менш надійний — для цього й потрібен `HAVING COUNT(*) >= 100`. "
     "Частково різницю AOV може пояснювати `freight_value`, що залежить від відстані до продавців, — це гіпотеза, яку варто перевірити окремим запитом.")

**5.3. Business-question:** Які категорії купували покупці з найбільшими витратами (`STRING_AGG`)?

In [ ]:
q53 = run("""
SELECT c.customer_unique_id,
       COUNT(DISTINCT o.order_id) AS n_orders,
       SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
       STRING_AGG(DISTINCT COALESCE(p.product_category_name, '(unknown)'), ', '
                  ORDER BY COALESCE(p.product_category_name, '(unknown)')) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 1
ORDER BY total_spend DESC NULLS LAST, c.customer_unique_id
LIMIT 20;
""")
display(q53)
q53["n_cat"] = q53.categories.str.count(",") + 1
t = q53.iloc[0]
show(f"**Interpretation:** Найбільший spend — {t.total_spend} ({n(t.n_orders)} зам.), категорії: {t.categories}. У топ-20 покупці в середньому охоплюють {q53.n_cat.mean():.1f} категорії, "
     f"{n((q53.n_orders > 1).sum())} із 20 мають більше одного замовлення — " + ("великі витрати переважно дає повторна купівля, а не одне дороге замовлення. " if (q53.n_orders > 1).sum() >= 10 else "великі витрати часто дає одне дороге замовлення, а не лояльність. ") +
     "`STRING_AGG(DISTINCT ... ORDER BY ...)` зводить набір категорій у зручну текстову ознаку; `COALESCE` не дає `NULL`-категорії зникнути зі списку.")

## 6. Set operations, window function, data-quality

**Вибір операторів.** `UNION` у 6.1 — потрібне *видалення дублікатів*: у `olist_customers` і `olist_sellers` кожен штат повторюється тисячі разів, і нам потрібні унікальні пари (state, role); `UNION ALL` залишив би їх усі. `INTERSECT` у 6.2 виражає «штат є в обох наборах» без join. `EXCEPT` у 6.4 — «є в одному, немає в іншому».

In [ ]:
q61 = run("""
SELECT customer_state AS state, 'customer' AS role FROM olist_customers
UNION
SELECT seller_state AS state, 'seller' AS role FROM olist_sellers
ORDER BY state, role;
""")
q62 = run("""
SELECT customer_state AS state FROM olist_customers
INTERSECT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state;
""")
q64 = run("""
SELECT customer_state AS state FROM olist_customers
EXCEPT
SELECT seller_state AS state FROM olist_sellers
ORDER BY state;
""")
display(q62.T)
show(f"**6.1 UNION:** {len(q61)} унікальних пар (state, role) замість сотень тисяч рядків — за рахунок `UNION` без `ALL`. "
     f"**6.2 INTERSECT:** штатів, де є і клієнти, і продавці — {len(q62)}. "
     f"**6.4 EXCEPT:** штати лише з клієнтами — {len(q64)}" + (f" ({', '.join(q64.state)})" if len(q64) else "") +
     ". Це узгоджується з FULL OUTER JOIN у 4.3 — той самий висновок, отриманий іншим інструментом.")

**6.3. Business-question:** Яке найновіше замовлення кожного реального покупця? (`ROW_NUMBER()`)

In [ ]:
q63 = run("""
WITH ranked_orders AS (
    SELECT c.customer_unique_id, o.customer_id, o.order_id, o.order_purchase_timestamp,
           ROW_NUMBER() OVER (
               PARTITION BY c.customer_unique_id
               ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
           ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
)
SELECT customer_unique_id, customer_id, order_id, order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;
""")
display(q63)
chk = run("""
WITH r AS (SELECT c.customer_unique_id,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC) AS rn
           FROM olist_orders o JOIN olist_customers c ON c.customer_id = o.customer_id)
SELECT COUNT(*) FILTER (WHERE rn = 1) AS latest_orders, COUNT(*) AS all_orders FROM r""").iloc[0]
show(f"**Interpretation:** `ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY ... DESC)` лишає по одному (найновішому) замовленню на покупця: {n(chk.latest_orders)} із {n(chk.all_orders)} замовлень. "
     "Додатковий ключ `order_id DESC` у `ORDER BY` робить вибір детермінованим, коли часові мітки збігаються. Це основа ознак «стан покупця на момент останнього замовлення».")

### 6.5. Data-quality перевірка (основа для Reflection)

In [ ]:
dq = run("""
SELECT
  (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL)                          AS products_no_category,
  (SELECT COUNT(*) FROM olist_products)                                                              AS products_total,
  (SELECT COUNT(*) FROM olist_orders o WHERE NOT EXISTS
        (SELECT 1 FROM olist_order_reviews r WHERE r.order_id = o.order_id))                         AS orders_without_review,
  (SELECT COUNT(*) - COUNT(DISTINCT review_id) FROM olist_order_reviews)                             AS duplicate_review_ids,
  (SELECT COUNT(*) FROM (SELECT order_id FROM olist_order_reviews GROUP BY order_id HAVING COUNT(*) > 1) x) AS orders_with_multiple_reviews,
  (SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE review_comment_title IS NULL OR TRIM(review_comment_title) = '') / COUNT(*), 1)
     FROM olist_order_reviews)                                                                       AS pct_empty_title,
  (SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE review_comment_message IS NULL OR TRIM(review_comment_message) = '') / COUNT(*), 1)
     FROM olist_order_reviews)                                                                       AS pct_empty_message,
  (SELECT COUNT(*) FROM olist_orders WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL) AS delivered_no_date,
  (SELECT COUNT(*) FROM olist_orders WHERE order_status NOT IN ('delivered'))                        AS non_delivered_orders;
""")
display(dq.T)
status = run("SELECT order_status, COUNT(*) AS n_orders FROM olist_orders GROUP BY order_status ORDER BY n_orders DESC")
display(status)
d = dq.iloc[0]
show(f"**Висновок.** Товарів без категорії: {n(d.products_no_category)} із {n(d.products_total)}; замовлень без review: {n(d.orders_without_review)}; "
     f"повторюваних `review_id`: {n(d.duplicate_review_ids)}; замовлень із кількома review: {n(d.orders_with_multiple_reviews)}; порожній заголовок відгуку — {d.pct_empty_title}%, порожній текст — {d.pct_empty_message}%; "
     f"`delivered` без дати доставки: {n(d.delivered_no_date)}; не-delivered замовлень: {n(d.non_delivered_orders)} (статус `{status.order_status[0]}` домінує — {n(status.n_orders[0])}).")

## Reflection

1. **Найскладніший JOIN** — `FULL OUTER JOIN`: потрібно тримати в голові «тільки зліва/справа» і склеювати ключ через `COALESCE`, інакше штат зникає з одного боку порівняння.
2. **Найважча business-question** — про seller-пари: пари продавців самі по собі мають слабкий зміст, тож довелося обмежити їх одним штатом і різними містами, щоб порівняння було регіональним, а не випадковим.
3. **Для production-ETL** найкорисніше `ON CONFLICT DO UPDATE` разом із `RETURNING`: перезапуск не дублює рядки, а результат одразу потрапляє в audit; constraints додатково не пускають у таблиці некоректні значення, навіть якщо помилку допустив upstream-скрипт.
4. **Window functions у ML:** `ROW_NUMBER()` для останнього замовлення покупця, `LAG()` для інтервалу між покупками, ковзні суми витрат за 30/90 днів, `RANK()` продавців усередині штату — з вікном, обмеженим моментом прогнозу, щоб майбутні замовлення не потрапили в ознаки (leakage).
5. **Якість даних:** частина товарів без категорії, замовлення без review, багато порожніх заголовків і текстів відгуків, повторювані `review_id` (тому в таблиці identity-ключ), а також не-delivered замовлення без дат доставки, які легко прийняти за «вчасні».
6. **`customer_unique_id`**, бо `customer_id` створюється на кожне замовлення: за ним frequency майже завжди дорівнює 1, і RFM втрачає зміст.
7. **Грануляція seller × order** потрібна, щоб відгук замовлення з кількома позиціями не враховувався кілька разів і не завищував `n_reviews`; лишається спрощення — review належить замовленню, а не продавцю.